# Task 5: Other BERT settings
In the first BERT notebook we used one setting.
Here we try two other settings and compare them on the validation part, not on the test data:
- a bigger learning rate (0.00005)
- DistilBERT: a smaller and faster version of BERT
We use Google Colab because the models need a GPU.

In [1]:
import torch
import transformers
import pandas as pd
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('GPU:', torch.cuda.is_available())
train = pd.read_csv('train_prep_v2.csv')
test = pd.read_csv('test_prep_v2.csv')
print(train.shape, test.shape)
print(train['label'].value_counts().to_dict())

torch: 2.14.1+cu130
transformers: 5.22.1
GPU: True
(4102, 9) (1026, 9)
{'ham': 3601, 'spam': 501}


## 1. Prepare the data
We prepare the data in the same way as the first BERT notebook:
the raw `text` column, the same validation part (same random seed),
and the same maximum length (64 tokens).
This way the comparison is fair.
DistilBERT should use the same word list as BERT. We check this.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

MAX_LEN = 64
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
tok_small = AutoTokenizer.from_pretrained('distilbert-base-uncased')
print('Same word list:', tok.get_vocab() == tok_small.get_vocab())

y_train = (train['label'] == 'spam').astype(int).values
y_test = (test['label'] == 'spam').astype(int).values
txt_fit, txt_val, y_fit, y_val = train_test_split(
    list(train['text']), y_train, test_size=0.2,
    random_state=42, stratify=y_train)
print('Fit  :', len(txt_fit), '| spam:', y_fit.sum())
print('Valid:', len(txt_val), '| spam:', y_val.sum())

def encode(texts):
    return tok(texts, max_length=MAX_LEN, truncation=True,
               padding='max_length', return_tensors='pt')

enc_fit = encode(txt_fit)
enc_val = encode(txt_val)
print(enc_fit['input_ids'].shape, enc_val['input_ids'].shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Same word list: True
Fit  : 3281 | spam: 401
Valid: 821 | spam: 100
torch.Size([3281, 64]) torch.Size([821, 64])


## 2. One function to train any model
We will try three settings. To keep everything the same,
we write one function that trains a model and gives back its result on the validation part.
- It starts with the same random seed every time.
- It uses early stopping: if the validation loss does not get better for 3 epochs, it stops.
  Then it goes back to the best epoch.
- The result is precision, recall and F1 on the validation part.

In [3]:
import copy
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score

device = torch.device('cuda')

def make_loader(enc, y, shuffle):
    data = TensorDataset(enc['input_ids'], enc['attention_mask'],
                         torch.tensor(y))
    return DataLoader(data, batch_size=32, shuffle=shuffle)

loader_fit = make_loader(enc_fit, y_fit, True)
loader_val = make_loader(enc_val, y_val, False)

def run_epoch(model, optimizer, loader, train):
    model.train(train)
    total_loss, n = 0, 0
    all_pred = []
    for ids, mask, labels in loader:
        ids = ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device)
        with torch.set_grad_enabled(train):
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
        if train:
            optimizer.zero_grad()
            out.loss.backward()
            optimizer.step()
        total_loss += out.loss.item() * len(labels)
        n += len(labels)
        all_pred.extend(out.logits.argmax(dim=1).cpu().numpy())
    return total_loss / n, np.array(all_pred)

def train_model(model_name, lr, name):
    torch.manual_seed(42)
    np.random.seed(42)
    torch.cuda.manual_seed_all(42)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    best_loss, best_epoch, best_state, wait = 999, 0, None, 0
    for epoch in range(1, 11):
        run_epoch(model, optimizer, loader_fit, True)
        va_loss, _ = run_epoch(model, optimizer, loader_val, False)
        if va_loss < best_loss:
            best_loss = va_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait == 3:
                break
    model.load_state_dict(best_state)
    _, pred = run_epoch(model, optimizer, loader_val, False)
    print(name, '| epochs:', epoch, '| best epoch:', best_epoch)
    row = [name, sum(p.numel() for p in model.parameters()), epoch, best_epoch,
           round(100 * precision_score(y_val, pred), 1),
           round(100 * recall_score(y_val, pred), 1),
           round(100 * f1_score(y_val, pred), 1)]
    del model, optimizer, best_state
    torch.cuda.empty_cache()
    return row

## 3. Train the three settings
- BERT with learning rate 0.00002: the same setting as the first BERT notebook.
- BERT with a bigger learning rate: 0.00005.
- DistilBERT with learning rate 0.00002: a smaller and faster version of BERT.
Every setting starts from the same random seed.
We compare them on the validation part.

In [4]:
rows = []
rows.append(train_model('bert-base-uncased', 2e-5, 'BERT lr 0.00002'))
rows.append(train_model('bert-base-uncased', 5e-5, 'BERT lr 0.00005'))
rows.append(train_model('distilbert-base-uncased', 2e-5, 'DistilBERT lr 0.00002'))

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT lr 0.00002 | epochs: 4 | best epoch: 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT lr 0.00002 | epochs: 6 | best epoch: 3


## 4. Compare the three settings
We put the three results in one table and save it.
We look at F1, because our data has much more ham than spam.

In [5]:
table = pd.DataFrame(rows, columns=['model', 'parameters', 'epochs',
                                    'best_epoch', 'precision', 'recall', 'f1'])
print(table.to_string(index=False))
table.to_csv('bert_variants.csv', index=False)
print('Saved')

                model  parameters  epochs  best_epoch  precision  recall   f1
      BERT lr 0.00002   109483778       4           1       98.9    93.0 95.9
      BERT lr 0.00005   109483778       5           2       96.9    94.0 95.4
DistilBERT lr 0.00002    66955010       6           3       98.9    93.0 95.9
Saved
